In [1]:
# ========================
# Torch and related
# ========================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
print(torch.__version__)

# ========================
# Standard libraries
# ========================
import os
import sys
import json
import random
from glob import glob
from collections import defaultdict

# ========================
# Numerical and Data Handling
# ========================
import numpy as np
import pandas as pd

# ========================
# Image I/O
# ========================
import imageio.v2 as imageio
import rasterio
from rasterio.transform import from_origin
from PIL import Image

# ========================
# Visualization
# ========================
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

# ========================
# Progress Bars
# ========================
from tqdm.notebook import tqdm  # Prefer notebook-friendly version

# ========================
# Geometry and Metrics
# ========================
from shapely.geometry import Polygon
from sklearn.metrics import jaccard_score, confusion_matrix

# ========================
# Segmentation Models (SMP)
# ========================
sys.path.append("D:/SEGMENTATION MODELS/segmentation_models_pytorch_main")
import segmentation_models_pytorch as smp

import warnings
import rasterio.errors

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

2.6.0+cu126


In [2]:
# ========================
# Model configuration
# ========================

#model_types = ['Segformer', 'Unet']  # Can be extended later
#encoder_names = ['mit_b5']
#encoder_names = ['mit_b5','resnet101', 'resnext101_32x8d']

# List of models to train
model_types = ['Segformer', 'DeepLabV3Plus', 'PSPNet', 'Unet']
#model_types = ['Segformer']  # Can be extended later
encoder_names = ['mit_b5']

# Map of dataset names to number of input channels
datasets = {
    "Planet_GT6": 48,
    "VV_GT6": 30,
    "VH_GT6": 30,
    "Combined_GT6": 108,
}

In [3]:
# ========================
# Training parameters
# ========================

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True

# Class weights to handle imbalance
weights = torch.tensor([1.0, 15.0, 35.0], dtype=torch.float32).to(DEVICE)

#LEARNING_RATE = 0.001
LEARNING_RATE =  0.0001

# Define loss and metric
loss = smp.utils.losses.CrossEntropyLoss(weight=weights)
metrics = [smp.utils.metrics.mIoU()]

In [4]:
# ========================
# Output and Dataset Setup
# ========================

output_dir = 'coffee_models_v5'
os.makedirs(output_dir, exist_ok=True)

results_df = pd.DataFrame(columns=['Dataset', 'Model', 'Encoder', 'IoU_0', 'IoU_1', 'IoU_2', 'mIoU', 'fwIoU'])


def label_transform(mask_np):
    return torch.from_numpy(mask_np).long()

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, target_paths, transform=None, transform_label=None):
        self.image_paths = image_paths
        self.target_paths = target_paths
        self.transform = transform
        self.transform_label = transform_label

    def __getitem__(self, index):
        with rasterio.open(self.image_paths[index]) as src:
            image = src.read().astype(np.float32)
            image = np.moveaxis(image, 0, -1)

        mask = imageio.imread(self.target_paths[index]).astype(np.int64)
        mask = np.where(mask == 2, 1, mask)
        mask = np.where(mask == 3, 2, mask)

        seed = np.random.randint(2147483647)
        random.seed(seed)
        torch.manual_seed(seed)

        if self.transform:
            image = self.transform(image)

        random.seed(seed)
        torch.manual_seed(seed)

        if self.transform_label:
            mask = self.transform_label(mask).squeeze(0)
        else:
            mask = torch.from_numpy(mask).long()  # ✅ fallback ensures safe shape

        return image, mask

    def __len__(self):
        return len(self.image_paths)

def evaluate_model(model, data_loader, device, num_classes=3):
    model.eval()
    total_conf_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)

    with torch.no_grad():
        for inp, lab in data_loader:
            inp, lab = inp.to(device), lab.to(device)
            outputs = model(inp)
            _, predictions = torch.max(outputs, dim=1)

            pred_flat = predictions.view(-1).cpu().numpy()
            lab_flat = lab.view(-1).cpu().numpy()

            total_conf_matrix += confusion_matrix(lab_flat, pred_flat, labels=range(num_classes))

        ious = []
        for i in range(num_classes):
            TP = total_conf_matrix[i, i]
            FP = np.sum(total_conf_matrix[:, i]) - TP
            FN = np.sum(total_conf_matrix[i, :]) - TP
            IoU = TP / float(TP + FP + FN) if (TP + FP + FN) > 0 else 0
            ious.append(IoU)

        mIoU = np.mean(ious)
        fwIoU = np.sum(np.sum(total_conf_matrix, axis=1) / np.sum(total_conf_matrix) * ious)

        return {'ious': ious, 'mIoU': mIoU, 'fwIoU': fwIoU}

def save_training_status(model_dir, epoch, best_loss, best_model_epoch):
    status_file = os.path.join(model_dir, 'train_status.json')
    status = {
        'last_completed_epoch': epoch,
        'best_loss': best_loss,
        'best_model_epoch': best_model_epoch
    }
    with open(status_file, 'w') as f:
        json.dump(status, f)

def load_training_status(model_dir):
    status_file = os.path.join(model_dir, 'train_status.json')
    if os.path.exists(status_file):
        with open(status_file, 'r') as f:
            return json.load(f)
    else:
        return None

In [5]:
# ========================
# Training and Evaluation Loop
# ========================

import gc

results_csv_path = os.path.join(output_dir, 'evaluation_results_seg2.csv')

# Load or initialize results
if os.path.exists(results_csv_path):
    results_df = pd.read_csv(results_csv_path, dtype=str)  # Load all as strings to preserve format
    print("📄 Loaded existing results:")
else:
    results_df = pd.DataFrame(columns=[
        'Dataset', 'Model', 'Encoder',
        'IoU_0', 'IoU_1', 'IoU_2',
        'mIoU', 'fwIoU'
    ])
    print("🆕 Starting new results table:")

display(results_df)

for dataset_name, input_channels in datasets.items():
    CLASSES = ['0', '1', '2']
    ACTIVATION = 'softmax2d'
    base_dir = f"D:/DATASETS/3_CAFE/{dataset_name}"

    train_imgs = sorted(glob(os.path.join(base_dir, "image_train/*.tiff")))
    train_masks = sorted(glob(os.path.join(base_dir, "class_train/*.png")))
    val_imgs = sorted(glob(os.path.join(base_dir, "image_val/*.tiff")))
    val_masks = sorted(glob(os.path.join(base_dir, "class_val/*.png")))
    test_imgs = sorted(glob(os.path.join(base_dir, "image_test/*.tiff")))
    test_masks = sorted(glob(os.path.join(base_dir, "class_test/*.png")))

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip()
    ])

    train_dataset = CustomDataset(train_imgs, train_masks, transform=transform, transform_label=transform)
    val_dataset = CustomDataset(val_imgs, val_masks, transform=transforms.ToTensor(), transform_label=None)
    test_dataset = CustomDataset(test_imgs, test_masks, transform=transforms.ToTensor(), transform_label=None)

    #train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0, pin_memory=True)
    #valid_loader = DataLoader(val_dataset, batch_size=10, shuffle=False, num_workers=0, pin_memory=True)
    #test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False, num_workers=0, pin_memory=True)

    train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True, num_workers=0, pin_memory=True)
    valid_loader = DataLoader(val_dataset, batch_size=20, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=20, shuffle=False, num_workers=0, pin_memory=True)

    for encoder_name in encoder_names:
        for model_type in model_types:
            already_evaluated = (
                (results_df['Dataset'] == dataset_name) &
                (results_df['Model'] == model_type) &
                (results_df['Encoder'] == encoder_name)
            ).any()

            if already_evaluated:
                print(f"✅ Already evaluated: {dataset_name} - {model_type} - {encoder_name}. Skipping.")
                clear_output(wait=True)
                display(results_df)
                continue

            print(f"\n🚀 Processing model: {model_type} with encoder: {encoder_name} on dataset: {dataset_name}")

            model = getattr(smp, model_type)(
                encoder_name=encoder_name,
                encoder_weights=None,
                classes=len(CLASSES),
                activation=ACTIVATION,
                in_channels=input_channels
            ).to(DEVICE)

            model_dir = os.path.join(output_dir, f"{dataset_name}_{model_type}_{encoder_name}")
            os.makedirs(model_dir, exist_ok=True)
            model_path = os.path.join(model_dir, f"{dataset_name}_{model_type}_{encoder_name}_best.pth")

            #optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


            #print(model.encoder.patch_embed1.proj)

            train_epoch = smp.utils.train.TrainEpoch(
                model,
                loss=loss,
                metrics=metrics,
                optimizer=optimizer,
                device=DEVICE,
                verbose=True,
            )

            valid_epoch = smp.utils.train.ValidEpoch(
                model,
                loss=loss,
                metrics=metrics,
                device=DEVICE,
                verbose=True,
            )

            max_iou = 0

            for i in range(50):
                print(f"\nEpoch: {i}")
                train_logs = train_epoch.run(train_loader)
                valid_logs = valid_epoch.run(valid_loader)
                print(f"Validation mIoU: {valid_logs['miou']:.4f}")

                if valid_logs['miou'] > max_iou:
                    max_iou = valid_logs['miou']
                    torch.save(model.state_dict(), model_path)
                    print("💾 Best Model Saved!")

                torch.cuda.empty_cache()

            print(f"✅ Loading best model from {model_path} for final evaluation...")
            model.load_state_dict(torch.load(model_path))
            model.to(DEVICE)

            results = evaluate_model(model, test_loader, DEVICE, num_classes=len(CLASSES))

            # Save individual IoUs in separate columns
            new_entry = pd.DataFrame([{
                'Dataset': dataset_name,
                'Model': model_type,
                'Encoder': encoder_name,
                'IoU_0': f"{results['ious'][0] * 100:.2f}",
                'IoU_1': f"{results['ious'][1] * 100:.2f}",
                'IoU_2': f"{results['ious'][2] * 100:.2f}",
                'mIoU': f"{results['mIoU'] * 100:.2f}",
                'fwIoU': f"{results['fwIoU'] * 100:.2f}"
            }])

            results_df = pd.concat([results_df, new_entry], ignore_index=True)
            results_df.drop_duplicates(subset=['Dataset', 'Model', 'Encoder'], keep='last', inplace=True)
            results_df.to_csv(results_csv_path, index=False)

            del model, optimizer, train_epoch, valid_epoch
            gc.collect()
            torch.cuda.empty_cache()

            clear_output(wait=True)
            display(results_df)

print(f"✅ Results continuously saved to {results_csv_path}")
print("🎉 Training and evaluation complete for all models, encoders, and datasets!")

,Dataset,Model,Encoder,IoU_0,IoU_1,IoU_2,mIoU,fwIoU
0,Planet_GT6,Segformer,mit_b5,96.32,83.91,66.63,82.29,94.20
1,VV_GT6,Segformer,mit_b5,91.23,65.36,17.04,57.88,86.75
2,VH_GT6,Segformer,mit_b5,91.82,68.24,23.35,61.13,87.73
3,Combined_GT6,Segformer,mit_b5,96.70,84.62,71.26,84.19,94.66
4,Planet_GT6,DeepLabV3Plus,mit_b5,96.56,85.56,58.15,80.09,94.62
5,VV_GT6,DeepLabV3Plus,mit_b5,87.72,61.39,6.72,51.94,83.13
6,VH_GT6,DeepLabV3Plus,mit_b5,90.88,67.15,12.28,56.77,86.72
7,Combined_GT6,DeepLabV3Plus,mit_b5,96.34,84.56,59.76,80.22,94.28
8,Planet_GT6,PSPNet,mit_b5,95.60,80.81,44.68,73.69,92.99
9,VV_GT6,PSPNet,mit_b5,90.74,66.15,14.66,57.18,86.45


✅ Results continuously saved to coffee_models_v5\evaluation_results_seg2.csv
🎉 Training and evaluation complete for all models, encoders, and datasets!


In [ ]:
# ========================
# Training and Evaluation Loop
# ========================

import gc

results_csv_path = os.path.join(output_dir, 'evaluation_results_seg2.csv')

# Load or initialize results
if os.path.exists(results_csv_path):
    results_df = pd.read_csv(results_csv_path, dtype=str)  # Load all as strings to preserve format
    print("📄 Loaded existing results:")
else:
    results_df = pd.DataFrame(columns=[
        'Dataset', 'Model', 'Encoder',
        'IoU_0', 'IoU_1', 'IoU_2',
        'mIoU', 'fwIoU'
    ])
    print("🆕 Starting new results table:")

display(results_df)

for dataset_name, input_channels in datasets.items():
    CLASSES = ['0', '1', '2']
    ACTIVATION = 'softmax2d'
    base_dir = f"D:/DATASETS/3_CAFE/{dataset_name}"

    train_imgs = sorted(glob(os.path.join(base_dir, "image_train/*.tiff")))
    train_masks = sorted(glob(os.path.join(base_dir, "class_train/*.png")))
    val_imgs = sorted(glob(os.path.join(base_dir, "image_val/*.tiff")))
    val_masks = sorted(glob(os.path.join(base_dir, "class_val/*.png")))
    test_imgs = sorted(glob(os.path.join(base_dir, "image_test/*.tiff")))
    test_masks = sorted(glob(os.path.join(base_dir, "class_test/*.png")))

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip()
    ])

    train_dataset = CustomDataset(train_imgs, train_masks, transform=transform, transform_label=transform)
    val_dataset = CustomDataset(val_imgs, val_masks, transform=transforms.ToTensor(), transform_label=None)
    test_dataset = CustomDataset(test_imgs, test_masks, transform=transforms.ToTensor(), transform_label=None)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0, pin_memory=True)
    valid_loader = DataLoader(val_dataset, batch_size=10, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False, num_workers=0, pin_memory=True)

    for encoder_name in encoder_names:
        for model_type in model_types:
            already_evaluated = (
                (results_df['Dataset'] == dataset_name) &
                (results_df['Model'] == model_type) &
                (results_df['Encoder'] == encoder_name)
            ).any()

            if already_evaluated:
                print(f"✅ Already evaluated: {dataset_name} - {model_type} - {encoder_name}. Skipping.")
                clear_output(wait=True)
                display(results_df)
                continue

            print(f"\n🚀 Processing model: {model_type} with encoder: {encoder_name} on dataset: {dataset_name}")

            model = getattr(smp, model_type)(
                encoder_name=encoder_name,
                encoder_weights=None,
                classes=len(CLASSES),
                activation=ACTIVATION,
                in_channels=input_channels
            ).to(DEVICE)

            model_dir = os.path.join(output_dir, f"{dataset_name}_{model_type}_{encoder_name}")
            os.makedirs(model_dir, exist_ok=True)
            model_path = os.path.join(model_dir, f"{dataset_name}_{model_type}_{encoder_name}_best.pth")

            optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

            train_epoch = smp.utils.train.TrainEpoch(
                model,
                loss=loss,
                metrics=metrics,
                optimizer=optimizer,
                device=DEVICE,
                verbose=True,
            )

            valid_epoch = smp.utils.train.ValidEpoch(
                model,
                loss=loss,
                metrics=metrics,
                device=DEVICE,
                verbose=True,
            )

            max_iou = 0

            for i in range(150):
                print(f"\nEpoch: {i}")
                train_logs = train_epoch.run(train_loader)
                valid_logs = valid_epoch.run(valid_loader)
                print(f"Validation mIoU: {valid_logs['miou']:.4f}")

                if valid_logs['miou'] > max_iou:
                    max_iou = valid_logs['miou']
                    torch.save(model.state_dict(), model_path)
                    print("💾 Best Model Saved!")

                torch.cuda.empty_cache()

            print(f"✅ Loading best model from {model_path} for final evaluation...")
            model.load_state_dict(torch.load(model_path))
            model.to(DEVICE)

            results = evaluate_model(model, test_loader, DEVICE, num_classes=len(CLASSES))

            # Save individual IoUs in separate columns
            new_entry = pd.DataFrame([{
                'Dataset': dataset_name,
                'Model': model_type,
                'Encoder': encoder_name,
                'IoU_0': f"{results['ious'][0] * 100:.2f}",
                'IoU_1': f"{results['ious'][1] * 100:.2f}",
                'IoU_2': f"{results['ious'][2] * 100:.2f}",
                'mIoU': f"{results['mIoU'] * 100:.2f}",
                'fwIoU': f"{results['fwIoU'] * 100:.2f}"
            }])

            results_df = pd.concat([results_df, new_entry], ignore_index=True)
            results_df.drop_duplicates(subset=['Dataset', 'Model', 'Encoder'], keep='last', inplace=True)
            results_df.to_csv(results_csv_path, index=False)

            del model, optimizer, train_epoch, valid_epoch
            gc.collect()
            torch.cuda.empty_cache()

            clear_output(wait=True)
            display(results_df)

print(f"✅ Results continuously saved to {results_csv_path}")
print("🎉 Training and evaluation complete for all models, encoders, and datasets!")